# Lesson 18: Securing AI Agents with Cryptographic Receipts

## Hands-on Notebook

Dis notebook go waka you through four exercises:

1. **Sign your first receipt** for agent tool call and confirm am.
2. **Change the receipt anyhow** and watch as confirmation fail.
3. **Make three-receipt chain** and check say chain dey correct.
4. **Put wrapper for Microsoft Agent Framework tool call** make every action dey carry receipt.

All cryptographic primitives na from well-maintained libraries wey dem import (`pynacl` for Ed25519, `jcs` for RFC 8785 canonical JSON, `hashlib` from Python standard library for SHA-256). The receipt logic na plain Python wey you fit read and change.

Run di cells one by one. Every section short and complete by itself.


## Setup

Install di two dependencies. Both get permisiv licenses (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Helper Utilities

Dis two helpers dem dey handle base64url encoding (wey no get padding) and SHA-256 hashing of any kain objects. Dem dey make sure say the rest of the notebook go focus only for the receipt logic sef.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Sign your first receipt

Imagine say our agent for **Contoso Travel** just check flights from Sydney go Los Angeles for one customer. We wan record dis tool call as signed receipt so dat one future auditor fit confirm am without to trust us.

### Step 1.1: Generate a signing key

For production, the agent signing key go dey for hardware security module (HSM), Azure Key Vault, or other protected store. But for dis lesson, we go generate one fresh key for memory.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: Build the receipt payload

Di payload get everything wey we want di receipt make e show: who act, which tool, wit which arguments, wetin come back, under which policy, and when. We dey hash di arguments and result instead make we put dem inside, so receipt no go leak sensitive tin.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: Sign and assemble the receipt

Tri step dem:

1. Canonicalize di payload use JCS so di two implementations wey dey produce di same logical receipt go produce byte-identical bytes.
2. Hash di canonical bytes with SHA-256.
3. Sign di hash with di Ed25519 private key.

Di signature den join di original payload to produce di final receipt.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: Confirm di receipt

Confirmation na wetin dey reverse di process. We go comot di signature, calculate di correct hash again, den check di signature with di public key wey dey inside di receipt.

Auditor wey dey do dis confirmation no need anything from us again except di receipt alone. No service to call, no key directory to ask, no trust wey dem gats get.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

You suppose see `Receipt is valid: True`. Di agent don produce im first cryptographically signed audit record.


## Section 2: Change di receipt

Di koko for receipts na sey dem go show if person don try change dem. Make we prove am.

We go change one character for di receipt and watch as verification no go pass.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Wetin jus happen?

Wen we change `policy_id`, di canonical bytes dem change. Di SHA-256 hash of dose bytes dem change. Di signature (wey bin dey ova di original hash) no match di new hash again. Verification correct return `False`.

No way to change any field for di receipt an still make am verify, unless di attacker get di private key. As long as di private key dey for key vault an di public key dey published, to tamper no go fit hide.

Try am yourself: change di `tool_name` or `agent_id` or `timestamp` for di cell wey dey above an run am again. Every change go give invalid receipt.


## Section 3: Chain receipts together

One receipt dey protect one action. Most agents dey do plenty actions. To make the whole sequence tamper-evident, we link each receipt to the one wey come before by putting the previous receipt hash inside the new receipt payload.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

If pesin comot or rearrange receipt, the chain go break for that exact point. To check any later receipt no go fit work because the `previous_receipt_hash` no go match the real hash of the receipt wey come before.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Now break di chain by tamper wit di middle receipt and re-verify. Di tampered receipt no pass im signature check, AND di next receipt no pass im chain-link check (because im `previous_receipt_hash` no longer match di modified middle receipt hash).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 still dey verify (e no change and e no get any previous receipt to depend on). Receipt 1 fail for im signature check because we change `tool_args_hash`. Receipt 2 fail for im chain-link check because im `previous_receipt_hash` na im original (wey dem don change) receipt 1 dem use calculate am.

Even if attacker come re-sign the changed receipt 1 (wey dem no fit do without private key), the chain-link mismatch for receipt 2 go still show say dem tamper am. To hide the change, attacker for re-sign every receipt from di place wey dem start change, and dat one go need private key.


## Section 4: Wrap an agent tool call with receipt signing

For real deployment, you no go want sey every agent author go dey remember to call `make_receipt`. You want make receipt signing dey automatic for every tool invocation.

Dis na di simplest pattern: one wrapper class wey fit carry any callable tool function come return one receipt-emitting version of am. Dis one fit work for any agent framework, including Microsoft Agent Framework (`agent_framework.foundry`).

If you never set up Microsoft Foundry project, the local mock wey dey below still show di pattern.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### How to Take Work with Microsoft Agent Framework

Di `ReceiptedTool` wrapper wey dey above no get problem with any framework. If you wan use am inside agent wey dem build with Microsoft Agent Framework, you go register di wrapped function as tool. Na sketch (you fit change di mock to real Microsoft Foundry tool registration):

```python
# Pseudocode wey dey show the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You be Contoso Travel agent ..."
#     tools=[receipted_lookup],   # na the wrapped tool, no be the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles for June.")
#
# # After wey e run, every tool call wey the agent make get one signed receipt:
# audit_chain = receipted_lookup.receipts
```

Di agent framework no need sabi anything about receipts. Receipt signing na wrapping wey dem put for tool, e no jam the framework. Na so you fit add provenance to the agent code wey already dey without to rewrite the agent.


## Recap and stretch challenge

You get:

- Generate one Ed25519 key pair.
- Build and sign receipt for one agent tool call.
- Check receipt offline using only the public key.
- Change one receipt and see verification fail.
- Build hash-chained sequence of three receipts.
- Change middle of chain and see both signature failure and chain-link failure.
- Wrap one tool function with automatic receipt signing.

**Stretch challenge.** Add `request_id` field (one UUID for distributed tracing) for receipt schema. Update `make_receipt` to put am, and check say receipts still verify end to end. Then change the field after signing and check say verification no go pass. This go make you understand how every byte for canonical encoding dey add to the signature.

**Important boundary.** Receipts prove three tins and only three tins: attribution (say this key sign this content), integrity (the content never change since dem sign am), and ordering (say this receipt come after that receipt). Dem no prove say the agent action correct, say the policy named for `policy_id` actually dem evaluate am, or say the agent follow every rule. Receipts na foundation. Governance na the system wey you build on top.

Read the lesson README again with that boundary for mind. The common mistake wey teams dey make with receipts na to think "we get receipts" mean "we dey govern." E no mean so. Receipts make agent behavior dey auditable. Dem no make am correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
